# VexarDrive Fleet Analysis

## Data Scientist Intern Assessment

### Objective

Using one week of fleet trip and telemetry data, this analysis develops:

1. A Driver Behaviour Dashboard to identify and score risky versus safe driving patterns.
2. A Vehicle Health Status Dashboard to identify vehicles showing potentially abnormal sensor signatures that may warrant maind limitations.

In [2]:
import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

In [3]:
DATA_PATH = Path("../data/raw/VEXAR_Fleet_Dataset_CANDIDATE_VERSION.xlsx")

print("Dataset path:", DATA_PATH)
print("Dataset exists:", DATA_PATH.exists())

Dataset path: ..\data\raw\VEXAR_Fleet_Dataset_CANDIDATE_VERSION.xlsx
Dataset exists: True


In [52]:
drivers = pd.read_excel(
    DATA_PATH,
    sheet_name="Drivers",
    header=2
)

vehicles = pd.read_excel(
    DATA_PATH,
    sheet_name="Vehicles",
    header=2
)

trips = pd.read_excel(
    DATA_PATH,
    sheet_name="Trips",
    header=2
)

telemetry = pd.read_excel(
    DATA_PATH,
    sheet_name="Telemetry",
    header=2
)

datasets = {
    "Drivers": drivers,
    "Vehicles": vehicles,
    "Trips": trips,
    "Telemetry": telemetry,
}

In [53]:
overview = pd.DataFrame(
    [
        {
            "Table": name,
            "Rows": df.shape[0],
            "Columns": df.shape[1],
        }
        for name, df in datasets.items()
    ]
)

overview

,Table,Rows,Columns
0,Drivers,30,8
1,Vehicles,30,8
2,Trips,450,14
3,Telemetry,12987,13


In [54]:
schema = []

for table_name, df in datasets.items():
    for column in df.columns:
        schema.append(
            {
                "Table": table_name,
                "Column": column,
                "Data_Type": str(df[column].dtype),
                "Non_Null": df[column].notna().sum(),
                "Missing": df[column].isna().sum(),
                "Unique": df[column].nunique(dropna=True),
            }
        )

schema = pd.DataFrame(schema)

schema

,Table,Column,Data_Type,Non_Null,Missing,Unique
0,Drivers,Driver_ID,str,30,0,30
1,Drivers,Driver_Name,str,30,0,28
2,Drivers,Age,int64,30,0,17
3,Drivers,Gender,str,30,0,2
4,Drivers,License_Experience_Years,int64,30,0,13
5,Drivers,Date_Joined_Fleet,str,30,0,30
6,Drivers,Primary_Vehicle_ID,str,30,0,30
7,Drivers,Home_Hub,str,30,0,8
8,Vehicles,Vehicle_ID,str,30,0,30
9,Vehicles,Vehicle_Type,str,30,0,1


In [55]:
drivers["Date_Joined_Fleet"] = pd.to_datetime(
    drivers["Date_Joined_Fleet"],
    errors="coerce"
)

vehicles["Registration_Date"] = pd.to_datetime(
    vehicles["Registration_Date"],
    errors="coerce"
)

vehicles["Last_Service_Date"] = pd.to_datetime(
    vehicles["Last_Service_Date"],
    errors="coerce"
)

trips["Trip_Date"] = pd.to_datetime(
    trips["Trip_Date"],
    errors="coerce"
)

telemetry["Timestamp"] = pd.to_datetime(
    telemetry["Timestamp"],
    errors="coerce"
)

In [8]:
for name, df in datasets.items():
    print(name)
    df.info()

Drivers
<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Driver_ID                 30 non-null     str  
 1   Driver_Name               30 non-null     str  
 2   Age                       30 non-null     int64
 3   Gender                    30 non-null     str  
 4   License_Experience_Years  30 non-null     int64
 5   Date_Joined_Fleet         30 non-null     str  
 6   Primary_Vehicle_ID        30 non-null     str  
 7   Home_Hub                  30 non-null     str  
dtypes: int64(2), str(6)
memory usage: 2.0 KB
Vehicles
<class 'pandas.DataFrame'>
RangeIndex: 30 entries, 0 to 29
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   Vehicle_ID                 30 non-null     str  
 1   Vehicle_Type               30 non-null     str  
 2   Make

In [56]:
missingness = pd.DataFrame(
    {
        name: df.isna().sum()
        for name, df in datasets.items()
    }
)

missingness

,Drivers,Vehicles,Trips,Telemetry
Accel_X_g,NaN,NaN,NaN,0.000
Accel_Y_g,NaN,NaN,NaN,0.000
Accel_Z_g,NaN,NaN,NaN,0.000
Age,0.000,NaN,NaN,NaN
Avg_Speed_kmph,NaN,NaN,0.000,NaN
Date_Joined_Fleet,0.000,NaN,NaN,NaN
Distance_KM,NaN,NaN,0.000,NaN
Driver_ID,0.000,NaN,0.000,0.000
Driver_Name,0.000,NaN,NaN,NaN
Duration_Min,NaN,NaN,0.000,NaN


In [10]:
for name, df in datasets.items():
    print(f"\n{name}")
    print("Shape:", df.shape)
    print("Duplicate rows:", df.duplicated().sum())
    print("\nMissing values:")
    print(df.isna().sum())


Drivers
Shape: (30, 8)
Duplicate rows: 0

Missing values:
Driver_ID                   0
Driver_Name                 0
Age                         0
Gender                      0
License_Experience_Years    0
Date_Joined_Fleet           0
Primary_Vehicle_ID          0
Home_Hub                    0
dtype: int64

Vehicles
Shape: (30, 8)
Duplicate rows: 0

Missing values:
Vehicle_ID                   0
Vehicle_Type                 0
Make                         0
Model                        0
Manufacture_Year             0
Registration_Date            0
Odometer_KM_Start_of_Week    0
Last_Service_Date            0
dtype: int64

Trips
Shape: (450, 14)
Duplicate rows: 0

Missing values:
Trip_ID            0
Driver_ID          0
Vehicle_ID         0
Trip_Date          0
Start_Time         0
End_Time           0
Duration_Min       0
Distance_KM        0
Avg_Speed_kmph     0
Max_Speed_kmph     0
Start_Latitude     0
Start_Longitude    0
End_Latitude       0
End_Longitude      0
dtype: int64



In [57]:
duplicates = pd.DataFrame(
    [
        {
            "Table": name,
            "Duplicate_Rows": df.duplicated().sum(),
        }
        for name, df in datasets.items()
    ]
)

duplicates

,Table,Duplicate_Rows
0,Drivers,0
1,Vehicles,0
2,Trips,0
3,Telemetry,0


In [58]:
referential_integrity = pd.DataFrame(
    [
        {
            "Relationship": "Telemetry → Trips",
            "Invalid_Records": (
                ~telemetry["Trip_ID"].isin(trips["Trip_ID"])
            ).sum(),
        },
        {
            "Relationship": "Trips → Drivers",
            "Invalid_Records": (
                ~trips["Driver_ID"].isin(drivers["Driver_ID"])
            ).sum(),
        },
        {
            "Relationship": "Trips → Vehicles",
            "Invalid_Records": (
                ~trips["Vehicle_ID"].isin(vehicles["Vehicle_ID"])
            ).sum(),
        },
    ]
)

referential_integrity

,Relationship,Invalid_Records
0,Telemetry → Trips,0
1,Trips → Drivers,0
2,Trips → Vehicles,0


In [59]:
telemetry_counts = telemetry.groupby("Trip_ID").size()

trip_coverage = trips[
    ["Trip_ID", "Duration_Min"]
].copy()

trip_coverage["Telemetry_Count"] = (
    trip_coverage["Trip_ID"].map(telemetry_counts)
)

trip_coverage["Telemetry_Count"] = (
    trip_coverage["Telemetry_Count"].fillna(0)
)

trip_coverage["Difference"] = (
    trip_coverage["Telemetry_Count"]
    - trip_coverage["Duration_Min"]
)

trip_coverage

,Trip_ID,Duration_Min,Telemetry_Count,Difference
0,T00001,16,16,0
1,T00002,22,22,0
2,T00003,27,27,0
3,T00004,42,42,0
4,T00005,13,13,0
...,...,...,...,...
445,T00446,29,29,0
446,T00447,16,16,0
447,T00448,25,25,0
448,T00449,32,32,0


In [60]:
coverage_summary = {
    "Trips": len(trip_coverage),
    "Trips_with_exact_coverage": (
        trip_coverage["Difference"] == 0
    ).sum(),
    "Trips_with_missing_telemetry": (
        trip_coverage["Telemetry_Count"] == 0
    ).sum(),
    "Trips_with_coverage_mismatch": (
        trip_coverage["Difference"] != 0
    ).sum(),
}

pd.Series(coverage_summary)

Trips                           450
Trips_with_exact_coverage       450
Trips_with_missing_telemetry      0
Trips_with_coverage_mismatch      0
dtype: int64

In [61]:
trips["Start_Datetime"] = pd.to_datetime(
    trips["Trip_Date"].dt.strftime("%Y-%m-%d")
    + " "
    + trips["Start_Time"].astype(str),
    errors="coerce"
)

trips["End_Datetime"] = pd.to_datetime(
    trips["Trip_Date"].dt.strftime("%Y-%m-%d")
    + " "
    + trips["End_Time"].astype(str),
    errors="coerce"
)

trips["Calculated_Duration_Min"] = (
    trips["End_Datetime"]
    - trips["Start_Datetime"]
).dt.total_seconds() / 60

trips["Duration_Difference_Min"] = (
    trips["Calculated_Duration_Min"]
    - trips["Duration_Min"]
)

trips[
    [
        "Trip_ID",
        "Duration_Min",
        "Calculated_Duration_Min",
        "Duration_Difference_Min",
    ]
].head()

,Trip_ID,Duration_Min,Calculated_Duration_Min,Duration_Difference_Min
0,T00001,16,16.000,0.000
1,T00002,22,22.000,0.000
2,T00003,27,27.000,0.000
3,T00004,42,42.000,0.000
4,T00005,13,13.000,0.000


In [62]:
duration_check = pd.Series(
    {
        "Trips": len(trips),
        "Exact_matches": (
            trips["Duration_Difference_Min"].abs() < 1e-9
        ).sum(),
        "Mismatches": (
            trips["Duration_Difference_Min"].abs() >= 1e-9
        ).sum(),
        "Negative_calculated_duration": (
            trips["Calculated_Duration_Min"] < 0
        ).sum(),
    }
)

duration_check

Trips                           450
Exact_matches                   450
Mismatches                        0
Negative_calculated_duration      0
dtype: int64

In [63]:
range_checks = pd.Series(
    {
        "Negative trip duration": (
            trips["Duration_Min"] < 0
        ).sum(),

        "Negative trip distance": (
            trips["Distance_KM"] < 0
        ).sum(),

        "Negative average speed": (
            trips["Avg_Speed_kmph"] < 0
        ).sum(),

        "Negative maximum speed": (
            trips["Max_Speed_kmph"] < 0
        ).sum(),

        "Maximum speed below average speed": (
            trips["Max_Speed_kmph"]
            < trips["Avg_Speed_kmph"]
        ).sum(),
    }
)

range_checks

Negative trip duration               0
Negative trip distance               0
Negative average speed               0
Negative maximum speed               0
Maximum speed below average speed    0
dtype: int64

In [64]:
gps_checks = pd.Series(
    {
        "Invalid trip start latitude": (
            ~trips["Start_Latitude"].between(-90, 90)
        ).sum(),

        "Invalid trip end latitude": (
            ~trips["End_Latitude"].between(-90, 90)
        ).sum(),

        "Invalid trip start longitude": (
            ~trips["Start_Longitude"].between(-180, 180)
        ).sum(),

        "Invalid trip end longitude": (
            ~trips["End_Longitude"].between(-180, 180)
        ).sum(),

        "Invalid telemetry latitude": (
            ~telemetry["Latitude"].between(-90, 90)
        ).sum(),

        "Invalid telemetry longitude": (
            ~telemetry["Longitude"].between(-180, 180)
        ).sum(),
    }
)

gps_checks

Invalid trip start latitude     0
Invalid trip end latitude       0
Invalid trip start longitude    0
Invalid trip end longitude      0
Invalid telemetry latitude      0
Invalid telemetry longitude     0
dtype: int64

In [65]:
driver_vehicle_check = trips.merge(
    drivers[
        [
            "Driver_ID",
            "Primary_Vehicle_ID",
        ]
    ],
    on="Driver_ID",
    how="left",
)

driver_vehicle_check["Uses_Primary_Vehicle"] = (
    driver_vehicle_check["Vehicle_ID"]
    == driver_vehicle_check["Primary_Vehicle_ID"]
)

driver_vehicle_check[
    "Uses_Primary_Vehicle"
].value_counts(dropna=False)

Uses_Primary_Vehicle
True     436
False     14
Name: count, dtype: int64

In [66]:
driver_vehicle_summary = pd.Series(
    {
        "Total trips": len(driver_vehicle_check),
        "Trips using primary vehicle": (
            driver_vehicle_check["Uses_Primary_Vehicle"]
        ).sum(),
        "Trips using another vehicle": (
            ~driver_vehicle_check["Uses_Primary_Vehicle"]
        ).sum(),
    }
)

driver_vehicle_summary

Total trips                    450
Trips using primary vehicle    436
Trips using another vehicle     14
dtype: int64

In [67]:
telemetry_identity = telemetry.merge(
    trips[
        [
            "Trip_ID",
            "Driver_ID",
            "Vehicle_ID",
        ]
    ],
    on="Trip_ID",
    how="left",
    suffixes=("_Telemetry", "_Trip"),
)

driver_mismatches = (
    telemetry_identity["Driver_ID_Telemetry"]
    != telemetry_identity["Driver_ID_Trip"]
).sum()

vehicle_mismatches = (
    telemetry_identity["Vehicle_ID_Telemetry"]
    != telemetry_identity["Vehicle_ID_Trip"]
).sum()

pd.Series(
    {
        "Driver ID mismatches": driver_mismatches,
        "Vehicle ID mismatches": vehicle_mismatches,
    }
)

Driver ID mismatches     0
Vehicle ID mismatches    0
dtype: int64

In [69]:
temporal_coverage = pd.Series(
    {
        "Trip start date": trips["Trip_Date"].min(),
        "Trip end date": trips["Trip_Date"].max(),
        "Telemetry start": telemetry["Timestamp"].min(),
        "Telemetry end": telemetry["Timestamp"].max(),
        "Number of trip dates": trips["Trip_Date"].nunique(),
    }
)

temporal_coverage

Trip start date         2026-07-31 00:00:00
Trip end date           2026-08-06 00:00:00
Telemetry start         2026-07-31 06:07:00
Telemetry end           2026-08-06 22:34:00
Number of trip dates                      7
dtype: object